In [ ]:
import sys
import pathlib

# set pythonpath to the main module directory
module_dir = pathlib.Path("..").parent.resolve().parent
if str(module_dir) not in sys.path:
    sys.path.append(str(module_dir))

In [ ]:
import pandas as pd


def load_results(clean_path: str, dirty_path: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    clean_df = pd.read_json(clean_path, orient="records")
    dirty_df = pd.read_json(dirty_path, orient="records")
    return clean_df, dirty_df


def preview_results(clean_df: pd.DataFrame, dirty_df: pd.DataFrame, sample_size: int = 10) -> None:
    if len(clean_df) > 0:
        display(clean_df.sample(min(sample_size, len(clean_df))))
    if len(dirty_df) > 0:
        display(dirty_df.sample(min(sample_size, len(dirty_df))))


clean_results_path = "../clean_results/greedy/res_clean.json"
dirty_results_path = "../logs/silent-norm-runs-v3/results.json"

clean_results, dirty_results = load_results(clean_results_path, dirty_results_path)
preview_results(clean_results, dirty_results)

In [ ]:
# Core preprocessing: normalize metric names + filter rows first, then pivot.

BENCHMARK_METRIC_SEP = " / "
BENCHMARKS_TO_REMOVE = ["blimp_", "hh-rlhf", "lmsys", "slim-orca"]
METRICS_TO_REMOVE = ["_raw", "score"]


def preprocess_results(df: pd.DataFrame) -> pd.DataFrame:
    required_columns = {"path", "benchmark", "metric", "value"}
    missing_columns = required_columns.difference(df.columns)
    if missing_columns:
        raise ValueError(f"Missing required columns: {sorted(missing_columns)}")

    out = df.copy()
    out["metric"] = out["metric"].astype(str).str.replace(",none", "", regex=False)

    remove_benchmark = pd.Series(False, index=out.index)
    for pattern in BENCHMARKS_TO_REMOVE:
        remove_benchmark = remove_benchmark | out["benchmark"].str.contains(pattern, na=False)

    remove_metric = pd.Series(False, index=out.index)
    for pattern in METRICS_TO_REMOVE:
        remove_metric = remove_metric | out["metric"].str.contains(pattern, na=False)

    out = out[~(remove_benchmark | remove_metric)].copy()
    out["run_id"] = out["path"].str.rsplit("/", n=1).str[0]

    if out["run_id"].isna().any():
        raise ValueError("run_id contains null values")
    if out["benchmark"].isna().any() or out["metric"].isna().any():
        raise ValueError("benchmark/metric contains null values")

    return out


def pivot_results(df: pd.DataFrame) -> pd.DataFrame:
    required_columns = {"run_id", "benchmark", "metric", "value"}
    missing_columns = required_columns.difference(df.columns)
    if missing_columns:
        raise ValueError(f"Missing required columns for pivot: {sorted(missing_columns)}")

    duplicate_keys = df.duplicated(subset=["run_id", "benchmark", "metric"], keep=False)
    if duplicate_keys.any():
        duplicate_preview = df.loc[duplicate_keys, ["run_id", "benchmark", "metric"]].head(10)
        raise ValueError(
            f"Found duplicate (run_id, benchmark, metric) rows; cannot pivot safely. Preview:\n{duplicate_preview.to_string(index=False)}"
        )

    per_metric_columns = {"benchmark", "metric", "value", "file"}
    metadata_columns = [c for c in df.columns if c not in per_metric_columns]

    for column_name in metadata_columns:
        bad_runs = df.groupby("run_id", dropna=False)[column_name].nunique(dropna=False)
        bad_runs = bad_runs[bad_runs > 1]
        if not bad_runs.empty:
            raise ValueError(f"Column '{column_name}' is not invariant within run_id. Example run_ids: {bad_runs.index[:10].tolist()}")

    pivoted = df.pivot(index="run_id", columns=["benchmark", "metric"], values="value")
    pivoted.columns = [f"{benchmark}{BENCHMARK_METRIC_SEP}{metric}" for benchmark, metric in pivoted.columns]
    pivoted = pivoted.reset_index()

    metadata = df[metadata_columns].drop_duplicates(subset=["run_id"])
    return metadata.merge(pivoted, on="run_id", how="inner", validate="one_to_one")


def prepare_pivoted_results(clean_df: pd.DataFrame, dirty_df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    clean_preprocessed = preprocess_results(clean_df)
    dirty_preprocessed = preprocess_results(dirty_df)

    clean_pivoted = pivot_results(clean_preprocessed)
    dirty_pivoted = pivot_results(dirty_preprocessed)
    return clean_pivoted, dirty_pivoted


clean_results, dirty_results = prepare_pivoted_results(clean_results, dirty_results)

print("Pivoted clean_results shape:", clean_results.shape)
print("Pivoted dirty_results shape:", dirty_results.shape)

In [ ]:
# Parse metadata from path columns on pivoted dataframes.


def add_path_metadata(clean_df: pd.DataFrame, dirty_df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    clean_out = clean_df.copy()
    dirty_out = dirty_df.copy()

    clean_parts = clean_out["path"].str.split("/")
    if (clean_parts.str.len() < 2).any():
        raise ValueError("Clean path does not contain enough segments to parse model_name")

    clean_out["model_name"] = clean_parts.str[-2]
    clean_out["train_dataset"] = pd.NA
    clean_out["layer_name"] = pd.NA
    clean_out["exp_name"] = pd.NA

    dirty_parts = dirty_out["path"].str.split("/")
    if (dirty_parts.str.len() < 5).any():
        raise ValueError("Dirty path does not contain enough segments to parse metadata")

    dirty_out["model_name"] = dirty_parts.str[-5]
    dirty_out["train_dataset"] = dirty_parts.str[-4]
    dirty_out["layer_name"] = dirty_parts.str[-3]
    dirty_out["exp_name"] = dirty_parts.str[-2]

    return clean_out, dirty_out


clean_results, dirty_results = add_path_metadata(clean_results, dirty_results)

if len(clean_results) > 0:
    display(clean_results.sample(min(10, len(clean_results))))
if len(dirty_results) > 0:
    display(dirty_results.sample(min(10, len(dirty_results))))

In [ ]:
def parse_experiment(exp_name: str) -> dict:
    """Parse experiment naming convention into optimization metadata.

    The parser supports the experiment tags currently used in this project,
    including baseline/learning-rate presets and explicit `kl=<value>` tags.

    Parameters
    ----------
    exp_name : str
        Experiment name string parsed from dirty-result paths.

    Returns
    -------
    dict
        Parsed fields with keys:
        - `kl_value` (float)
        - `lr_value` (float)
        - `early_stop` (bool)

    Raises
    ------
    ValueError
        If the experiment name does not match any supported pattern or if
        a required KL value cannot be extracted.
    """
    kl_value = None
    lr_value = None
    early_stop = True

    if "no-early-stop" in exp_name:
        early_stop = False

    if "baseline" in exp_name or "no-early-stop" in exp_name:
        lr_value = 0.1
        kl_value = 1.0

    if "small-lr" in exp_name:
        lr_value = 0.02
        kl_value = 1.0

    if "medium-lr" in exp_name:
        lr_value = 0.04
        kl_value = 1.0

    if "large-lr" in exp_name:
        lr_value = 0.25
        kl_value = 1.0

    if "small-kl" in exp_name:
        lr_value = 0.1
        kl_value = 0.5

    if "high-kl" in exp_name:
        lr_value = 0.1
        kl_value = 2.0

    if "-KL-" in exp_name:
        lr_value = 0.1

        parts = [part for part in exp_name.split("-")]
        idx = next((i for i, part in enumerate(parts) if part.startswith("KL")), None)
        if idx is None:
            raise ValueError(f"Could not find KL part in experiment name: {exp_name}")

        kl_part = parts[idx + 1]
        kl_value = float(kl_part)

    if "kl=" in exp_name:
        lr_value = 0.1

        kl_part = [part for part in exp_name.split("_") if part.startswith("kl=")]
        if not kl_part:
            raise ValueError(f"Could not find kl value in experiment name: {exp_name}")

        kl_value_str = kl_part[0].split("=")[1].split("-")[0]
        kl_value = float(kl_value_str)

    if lr_value is None or kl_value is None:
        raise ValueError(f"Could not parse experiment name: {exp_name}")

    return {"kl_value": kl_value, "lr_value": lr_value, "early_stop": early_stop}


def add_experiment_metadata(dirty_df: pd.DataFrame) -> pd.DataFrame:
    parsed_exp = dirty_df["exp_name"].apply(lambda x: pd.Series(parse_experiment(x)))
    out = dirty_df.copy()
    for column_name in parsed_exp.columns:
        if column_name in out.columns:
            out = out.drop(columns=[column_name])
    out = pd.concat([out, parsed_exp], axis=1)
    return out


dirty_results = add_experiment_metadata(dirty_results)

if len(dirty_results) > 0:
    display(dirty_results.sample(min(10, len(dirty_results))))

In [ ]:
# Metric names were normalized before pivoting.
# Keep this cell as a lightweight sanity check.


def get_metric_columns(df: pd.DataFrame) -> list[str]:
    return [c for c in df.columns if BENCHMARK_METRIC_SEP in c]


def print_metric_sanity(df: pd.DataFrame, df_name: str) -> list[str]:
    metric_columns = get_metric_columns(df)
    if not metric_columns:
        raise ValueError(f"No pivoted metric columns found in {df_name}")

    print(f"Pivoted metric columns in {df_name}:", len(metric_columns))
    print("Example metric columns:", metric_columns[:8])
    return metric_columns


metric_columns = print_metric_sanity(dirty_results, "dirty_results")

## Filtering

In [ ]:
# Print the column names of the clean and dirty metrics in a readable manner.

metric_columns_clean = get_metric_columns(clean_results)
metric_columns_dirty = get_metric_columns(dirty_results)


def print_metric_columns(clean_columns: list[str], dirty_columns: list[str]) -> None:
    print("==== Clean Metric Columns")
    for col in clean_columns:
        print(f"- {col}")

    print()
    print("==== Dirty Metric Columns")
    for col in dirty_columns:
        print(f"- {col}")


print_metric_columns(metric_columns_clean, metric_columns_dirty)

In [ ]:
# Scatter correlations: one figure per metric, with a subplot per against column.

import math
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


def _as_series(obj: pd.Series | pd.DataFrame) -> pd.Series:
    # If duplicate column names produce a DataFrame on selection, keep the first column.
    if isinstance(obj, pd.DataFrame):
        return obj.iloc[:, 0]
    return obj


def _safe_corr_xy(x: pd.Series, y: pd.Series) -> float:
    pair = pd.concat([x, y], axis=1, keys=["x", "y"]).dropna()
    if len(pair) < 2:
        return np.nan
    if pair["x"].nunique(dropna=True) < 2 or pair["y"].nunique(dropna=True) < 2:
        return np.nan
    return float(pair["x"].corr(pair["y"]))


def _plot_group_line_with_fixed_slope(ax, x: pd.Series, y: pd.Series, slope: float, color: str) -> None:
    pair = pd.concat([x, y], axis=1, keys=["x", "y"]).dropna()
    if len(pair) < 2 or pd.isna(slope):
        return

    x_vals = pair["x"].astype(float)
    y_vals = pair["y"].astype(float)
    if x_vals.nunique(dropna=True) < 2:
        return

    x_mean = x_vals.mean()
    y_mean = y_vals.mean()

    x_line = np.linspace(x_vals.min(), x_vals.max(), 100)
    y_line = y_mean + slope * (x_line - x_mean)
    ax.plot(x_line, y_line, color=color, linewidth=2, alpha=0.9)


def plot_metrics_correlation(
    dirty_df: pd.DataFrame,
    metric_columns: list[str],
    against_columns: list[str] = [
        "eval-oasst2 / top1_acc",
        "eval-oasst2 / top10_agr",
        "eval-oasst2 / kl_div",
        # "eval-tulu-v3 / top1_acc",
        # "eval-tulu-v3 / top10_agr",
        # "eval-tulu-v3 / kl_div",
    ],
    hue_column: str = "model_name",
) -> None:
    if hue_column not in dirty_df.columns:
        raise ValueError(f"Missing hue column: {hue_column}")

    missing_against = [col for col in against_columns if col not in dirty_df.columns]
    if missing_against:
        raise ValueError(f"Missing against columns: {missing_against}")

    valid_metric_columns = [col for col in metric_columns if col in dirty_df.columns]
    if not valid_metric_columns:
        raise ValueError("No valid metric columns were found in dataframe")

    # Draw one figure per metric; each figure contains subplots for against_columns.
    for metric in valid_metric_columns:
        subplot_count = len(against_columns)
        fig, axes = plt.subplots(1, subplot_count, figsize=(7 * subplot_count, 5), squeeze=False)
        axes = axes.flatten()

        for ax, against_col in zip(axes, against_columns):
            if metric == against_col:
                ax.text(0.5, 0.5, f"Skipping self-correlation: {metric}", ha="center", va="center")
                ax.set_axis_off()
                continue

            grouped = dirty_df.groupby(hue_column, dropna=False)
            group_corrs: dict[str, float] = {}

            unique_groups = list(grouped.groups.keys())
            palette = sns.color_palette("tab10", n_colors=max(1, len(unique_groups)))
            color_by_group = {group_name: palette[idx % len(palette)] for idx, group_name in enumerate(unique_groups)}

            for group_name, group_df in grouped:
                x = _as_series(group_df[against_col])
                y = _as_series(group_df[metric])

                valid = pd.concat([x, y], axis=1, keys=["x", "y"]).dropna()
                if valid.empty:
                    group_corrs[str(group_name)] = np.nan
                    continue

                color = color_by_group[group_name]
                ax.scatter(valid["x"], valid["y"], s=28, alpha=0.75, color=color)
                group_corrs[str(group_name)] = _safe_corr_xy(valid["x"], valid["y"])

            valid_corrs = [corr for corr in group_corrs.values() if pd.notna(corr)]
            avg_corr = float(np.mean(valid_corrs)) if valid_corrs else np.nan

            # for group_name, group_df in grouped:
            #     x = _as_series(group_df[against_col])
            #     y = _as_series(group_df[metric])
            #     _plot_group_line_with_fixed_slope(
            #         ax=ax,
            #         x=x,
            #         y=y,
            #         slope=avg_corr,
            #         color=color_by_group[group_name],
            #     )

            legend_handles = []
            for group_name in unique_groups:
                corr_val = group_corrs.get(str(group_name), np.nan)
                if pd.notna(corr_val):
                    label = f"{group_name} (corr={corr_val:.2f})"
                else:
                    label = f"{group_name} (corr=NA)"

                handle = plt.Line2D(
                    [0],
                    [0],
                    marker="o",
                    color=color_by_group[group_name],
                    markerfacecolor=color_by_group[group_name],
                    markersize=6,
                    linewidth=2,
                    label=label,
                )
                legend_handles.append(handle)

            avg_label = f"Avg corr={avg_corr:.2f}" if pd.notna(avg_corr) else "Avg corr=NA"
            ax.set_title(f"{metric} vs {against_col}\n{avg_label}")
            ax.set_xlabel(against_col)
            ax.set_ylabel(metric)
            ax.grid(alpha=0.3)
            ax.legend(handles=legend_handles, title=hue_column, fontsize=8, title_fontsize=9)

        fig.suptitle(f"Metric: {metric}", y=1.02, fontsize=12, fontweight="bold")
        plt.tight_layout()
        plt.show()


# plot_metrics_correlation(dirty_results, metric_columns_dirty)

In [ ]:
def filter_by_degradation(clean_results: pd.DataFrame, dirty_results: pd.DataFrame, columns: list[str], threshold: list[float] | float) -> pd.DataFrame:
    """
    Filters the dirty_results DataFrame to include only rows where the degradation in specified metric columns is at most the given threshold
    Degradation is computed via the formula: (clean_value - dirty_value) 
    Rows in which clean_value or dirty_value is missing for any of the specified do not effect the results, i.e. they are included in the output.
    
    Notes:
        - If any of the metrics do not appear in the clean_results or dirty_results then we skip this metric with a warning
        - The values should be matched via model_name column. 
          A single value in the clean_results is matched to all dirty_results rows with the same metadata, 
          and the degradation is computed for each matched row. 
    """
    if isinstance(threshold, float):
        threshold = [threshold] * len(columns)
    elif len(threshold) != len(columns):
        raise ValueError("Length of threshold list must match length of columns list")

    filtered_dirty = dirty_results.copy()

    keep_mask = pd.Series(True, index=filtered_dirty.index)
    
    for col, thresh in zip(columns, threshold):
        if col not in clean_results.columns:
            print(f"Warning: Column '{col}' not found in clean_results; skipping this metric.")
            continue
        if col not in dirty_results.columns:
            print(f"Warning: Column '{col}' not found in dirty_results; skipping this metric.")
            continue

        merged = pd.merge(
            filtered_dirty,
            clean_results[["model_name", col]],
            on=["model_name"],
            how="left",
            suffixes=("", "_clean"),
        )
                
        merged["degradation"] = merged[f"{col}_clean"] - merged[col]
        keep_mask = keep_mask & ~(merged["degradation"] > thresh)
        
    filtered_dirty = filtered_dirty[keep_mask]

    return filtered_dirty



def filter_by_improvement(clean_results: pd.DataFrame, dirty_results: pd.DataFrame, columns: list[str], threshold: list[float] | float) -> pd.DataFrame:
    """
    Filters the dirty_results DataFrame to include only rows where the improvement in at least one of the specified metric columns is at least the given threshold
    Improvement is computed via the formula: (dirty_value - clean_value) 
    Rows in which clean_value or dirty_value is missing for any of the specified do not effect the results.
    
    Notes:
        - If any of the metrics do not appear in the clean_results or dirty_results then we skip this metric with a warning
        - The values should be matched via model_name column. 
          A single value in the clean_results is matched to all dirty_results rows with the same metadata, 
          and the improvement is computed for each matched row. 
    """
    if isinstance(threshold, float):
        threshold = [threshold] * len(columns)
    elif len(threshold) != len(columns):
        raise ValueError("Length of threshold list must match length of columns list")

    filtered_dirty = dirty_results.copy()
    
    
    improvement_mask = pd.Series(False, index=filtered_dirty.index)
    for col, thresh in zip(columns, threshold):
        if col not in clean_results.columns:
            print(f"Warning: Column '{col}' not found in clean_results; skipping this metric.")
            continue
        if col not in dirty_results.columns:
            print(f"Warning: Column '{col}' not found in dirty_results; skipping this metric.")
            continue

        merged = pd.merge(
            filtered_dirty,
            clean_results[["model_name", col]],
            on=["model_name"],
            how="left",
            suffixes=("", "_clean"),
        )
                
        merged["improvement"] = merged[col] - merged[f"{col}_clean"]
        improvement_mask = improvement_mask | (merged["improvement"] >= thresh)
        
        
    filtered_dirty = filtered_dirty[improvement_mask]
        
    return filtered_dirty
        



In [ ]:
KNOWN_METRIC_KEYWORDS = ("acc", "f1", "match", "validity", "compliance", "pass")


def is_known_range_metric(metric_name: str) -> bool:
    metric_name = str(metric_name).lower()
    return any(keyword in metric_name for keyword in KNOWN_METRIC_KEYWORDS)

BENCHMARK_COLUMNS = [m for m in clean_results.columns if is_known_range_metric(m)]

In [ ]:
dirty_results_filtered = dirty_results.copy()

dirty_results_filtered = filter_by_degradation(clean_results, dirty_results, columns=BENCHMARK_COLUMNS, threshold=0.04)

# dirty_results_filtered = filter_by_improvement(clean_results, dirty_results, columns=BENCHMARK_COLUMNS, threshold=0.2)


In [ ]:
dirty_results_filtered = dirty_results_filtered[
    (dirty_results_filtered["kl_value"] == 1.0) &  
    # (dirty_results_filtered["eval-oasst2 / proj_l2_rel"] >= 0.1) & 
#     (dirty_results_filtered["eval-oasst2 / top1_acc"] >= 0.95) &
#     (dirty_results_filtered["eval-oasst2 / top10_agr"] >= 0.9) & 
    (dirty_results_filtered["eval-tulu-v3 / proj_l2_rel"] >= 0.1)
#     (dirty_results_filtered["eval-tulu-v3 / top1_acc"] >= 0.95) &
#     (dirty_results_filtered["eval-tulu-v3 / top10_agr"] >= 0.9)
]


print(f"Number of runs before filtering: {len(dirty_results)}")
print(f"Number of runs after filtering: {len(dirty_results_filtered)}")
display(dirty_results_filtered[["model_name", "train_dataset", "layer_name"]].drop_duplicates())

## Plotting

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

KNOWN_METRIC_KEYWORDS = ("acc", "f1", "match", "validity", "compliance", "pass")
PLOT_COLUMNS = [
    "benchmark",
    "metric",
    "benchmark_metric",
    "value_dirty",
    "value_clean",
    "value_plot",
    "is_known_range",
]


def is_known_range_metric(metric_name: str) -> bool:
    metric_name = str(metric_name).lower()
    return any(keyword in metric_name for keyword in KNOWN_METRIC_KEYWORDS)


def resolve_clean_row(clean_df: pd.DataFrame, model_name: str) -> pd.Series:
    model_rows = clean_df[clean_df["model_name"] == model_name]
    if model_rows.empty:
        raise ValueError(f"No clean row found for model_name='{model_name}'")
    if len(model_rows) > 1:
        raise ValueError(f"Expected one clean row for model_name='{model_name}', got {len(model_rows)}")
    return model_rows.iloc[0]


def as_float_or_none(value, column_name: str, source_name: str) -> float | None:
    if pd.isna(value):
        return None
    try:
        return float(value)
    except (TypeError, ValueError) as exc:
        raise ValueError(f"Non-numeric value in {source_name} column '{column_name}': {value!r}") from exc


def build_plot_frames(dirty_row: pd.Series, clean_row: pd.Series) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    empty = pd.DataFrame(columns=PLOT_COLUMNS)

    dirty_metric_columns = [c for c in dirty_row.index if BENCHMARK_METRIC_SEP in str(c)]
    clean_metric_columns = set(c for c in clean_row.index if BENCHMARK_METRIC_SEP in str(c))

    common_rows = []
    dirty_only_rows = []

    for column_name in dirty_metric_columns:
        benchmark_name, metric_name = column_name.split(BENCHMARK_METRIC_SEP, 1)
        dirty_value = as_float_or_none(dirty_row[column_name], column_name, "dirty")

        if column_name in clean_metric_columns:
            clean_value = as_float_or_none(clean_row[column_name], column_name, "clean")
            if dirty_value is None or clean_value is None:
                continue

            known_range = is_known_range_metric(metric_name)
            value_plot = (dirty_value - clean_value) * 100.0 if known_range else (dirty_value - clean_value)
            common_rows.append(
                {
                    "benchmark": benchmark_name,
                    "metric": metric_name,
                    "benchmark_metric": column_name,
                    "value_dirty": dirty_value,
                    "value_clean": clean_value,
                    "value_plot": value_plot,
                    "is_known_range": known_range,
                }
            )
        else:
            if dirty_value is None:
                continue

            dirty_only_rows.append(
                {
                    "benchmark": benchmark_name,
                    "metric": metric_name,
                    "benchmark_metric": column_name,
                    "value_dirty": dirty_value,
                    "value_clean": pd.NA,
                    "value_plot": dirty_value,
                    "is_known_range": False,
                }
            )

    if common_rows:
        common_df = pd.DataFrame(common_rows).sort_values("benchmark_metric").reset_index(drop=True)
        known_df = common_df[common_df["is_known_range"]].copy().reset_index(drop=True)
        unknown_df = common_df[~common_df["is_known_range"]].copy().reset_index(drop=True)
    else:
        known_df = empty.copy()
        unknown_df = empty.copy()

    if dirty_only_rows:
        dirty_only_df = pd.DataFrame(dirty_only_rows).sort_values("benchmark_metric").reset_index(drop=True)
        dirty_only_df = dirty_only_df[PLOT_COLUMNS]
    else:
        dirty_only_df = empty.copy()

    return known_df, unknown_df, dirty_only_df


def plot_panel(ax, df: pd.DataFrame, ylabel: str, panel_title: str, label_mode: str, fontsize: int = 10) -> None:
    ax.set_title(panel_title, pad=12)
    ax.set_xlabel("")
    ax.set_ylabel(ylabel)
    ax.tick_params(axis="x", rotation=90, labelsize=fontsize)
    ax.tick_params(axis="y", labelsize=fontsize)
    ax.grid(axis="y", alpha=0.3)

    colors = ["#d62728" if value < 0 else "#1f77b4" for value in df["value_plot"]]
    ax.bar(df["benchmark_metric"], df["value_plot"], color=colors)

    for idx, bar in enumerate(ax.patches):
        if idx >= len(df):
            break

        dirty_value = df.iloc[idx]["value_dirty"]
        clean_value = df.iloc[idx]["value_clean"]

        if label_mode == "percent":
            label = f"{(dirty_value - clean_value) * 100:.1f}"
        elif label_mode == "diff":
            label = f"{dirty_value:.3f} - {clean_value:.3f}"
        elif label_mode == "raw":
            label = f"{dirty_value:.3f}"
        else:
            raise ValueError(f"Unsupported label_mode: {label_mode}")

        x_center = bar.get_x() + bar.get_width() / 2
        y_anchor = max(bar.get_height(), 0)
        ax.annotate(label, xy=(x_center, y_anchor), xytext=(0, 2), textcoords="offset points", ha="center", va="bottom", fontsize=fontsize)

In [ ]:
def plot_run_comparisons(dirty_filtered_df: pd.DataFrame, clean_df: pd.DataFrame) -> None:
    run_ids = dirty_filtered_df["run_id"].tolist()
    print("Number of runs to plot:", len(run_ids))

    for _, dirty_row in dirty_filtered_df.iterrows():
        run_id = dirty_row["run_id"]
        model_name = dirty_row["model_name"]
        train_dataset = dirty_row["train_dataset"]
        layer_name = dirty_row["layer_name"]

        clean_row = resolve_clean_row(clean_df, model_name=model_name)
        known_df, unknown_df, dirty_only_df = build_plot_frames(dirty_row, clean_row)

        panel_specs = []
        if not known_df.empty:
            panel_specs.append(
                {
                    "df": known_df,
                    "ylabel": "Difference % (for range [0-1] metrics)",
                    "title": "Known-range metrics: (dirty - clean) x 100",
                    "label_mode": "percent",
                }
            )
        if not unknown_df.empty:
            panel_specs.append(
                {
                    "df": unknown_df,
                    "ylabel": "Difference (raw)",
                    "title": "Unknown-range metrics: (dirty - clean)",
                    "label_mode": "diff",
                }
            )
        if not dirty_only_df.empty:
            panel_specs.append(
                {
                    "df": dirty_only_df,
                    "ylabel": "Value (raw)",
                    "title": "Dirty-only metrics (not present in clean)",
                    "label_mode": "raw",
                }
            )

        if not panel_specs:
            continue

        fig, axes = plt.subplots(len(panel_specs), 1, figsize=(18, 5 * len(panel_specs)))
        if len(panel_specs) == 1:
            axes = [axes]

        fig.suptitle(
            f"Model: {model_name} | Dataset: {train_dataset} | Layer: {layer_name} | KL-Weight: {dirty_row.get('kl_value', 'N/A')}\n Run ID: {dirty_row.get('run_id', 'N/A')}",
            y=1.04,
            fontsize=11,
            fontweight="bold",
        )

        for ax, spec in zip(axes, panel_specs):
            plot_panel(
                ax=ax,
                df=spec["df"],
                ylabel=spec["ylabel"],
                panel_title=spec["title"],
                label_mode=spec["label_mode"],
                fontsize=10,
            )

        plt.tight_layout()
        plt.show()
        


plot_run_comparisons(dirty_results_filtered, clean_results)